In [ ]:
import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:
stiffness_data_collection = []

In [ ]:
hessianShiftForStiffness = 1e-7


### Pattern 1: Parallel tubes

In [ ]:
angle = 90
r = 2.5
avg_len = 0.08


h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
m, marker = pattern_generator_using_gmsh.get_single_dash(h, avg_len, avg_len, dash_point = dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

# vertices = m.vertices()
# vertices *= 4
# m = MeshFEM.mesh.Mesh(vertices, m.elements())
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()


In [ ]:
name = 'parallel_tubes'

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.recordStart("{}_inflation.mp4".format(name), outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
import experiment_helper

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
# ipu.setVars(np.load('parallel_tubes.npy'))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()
benchmark.report()

In [ ]:
viewer.recordStop()

### Pattern 2: Dashline

In [ ]:
a = 2
avg_len = 0.08
radius = 0.8

In [ ]:
angle = 45
r = 0.9

In [ ]:

h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
m, marker = pattern_generator_using_gmsh.get_single_dash(h, avg_len, avg_len, dash_point = dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

# vertices = m.vertices()
# vertices *= 4
# m = MeshFEM.mesh.Mesh(vertices, m.elements())
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
boundary_vxs, boundary_lines, dash_line, curve_edges = pattern_generator_using_gmsh.get_single_dash(h, avg_len, avg_len, dash_point = dash_point, return_line_segments = True)

x = dash_line[:, [0]]
y = dash_line[:, [1]]
mirror_right_x = 5 - x
mirror_up_y = 5 - y
z = dash_line[:, [2]]

total_curves = np.concatenate((dash_line, np.concatenate((x, mirror_up_y, z), axis = 1), np.concatenate((mirror_right_x, y, z), axis = 1), np.concatenate((mirror_right_x, mirror_up_y, z), axis = 1)))

total_edges = np.concatenate((curve_edges - 1, curve_edges + len(dash_line) - 1, curve_edges + 2 * len(dash_line) - 1, curve_edges + 3 * len(dash_line) - 1))

visualization.plot_line_segments(total_curves, total_edges, color = 'black')
low = -2.5
high = 7.5
plt.plot([low, low], [low, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([low, high], [low, low], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([low, high], [high, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([high, high], [low, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.axis('off')
plt.savefig('dash_line.svg', dpi = 300, bbox_inches='tight')

In [ ]:
plt.clf()
plt.close()

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
visualization.plot_2d_mesh(m, pointList = fusedVtx, width = 10, height = 10)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
name = 'dash_line'

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.recordStart("{}_inflation.mp4".format(name), outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
import experiment_helper

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
# ipu.setVars(np.load('parallel_tubes.npy'))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()
benchmark.report()

In [ ]:
viewer.recordStop()

### Pattern 3: Cosine curve

In [ ]:
use_half_period = False
h = 5
m, marker = pattern_generator_using_gmsh.get_cosine_curve(h, avg_len, avg_len, amplitude=0.1653061224489796, end_threshold = 0.0, use_half_period=use_half_period)

finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

# vertices = m.vertices()
# vertices *= 4
# m = MeshFEM.mesh.Mesh(vertices, m.elements())
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)


In [ ]:
base_cosine_curve, curve_edges, boundary_vxs, boundary_lines = pattern_generator_using_gmsh.get_cosine_curve(h, avg_len, avg_len, amplitude=0.1653061224489796, end_threshold = 0.0, use_half_period=use_half_period, return_line_segments=True)

x = base_cosine_curve[:, [0]]
y = base_cosine_curve[:, [1]]
mirror_right_x = 5 - x
mirror_up_y = 5 - y
z = base_cosine_curve[:, [2]]

total_curves = np.concatenate((base_cosine_curve, np.concatenate((x, mirror_up_y, z), axis = 1), np.concatenate((mirror_right_x, y, z), axis = 1), np.concatenate((mirror_right_x, mirror_up_y, z), axis = 1)))

total_edges = np.concatenate((curve_edges - 1, curve_edges + len(base_cosine_curve) - 1, curve_edges + 2 * len(base_cosine_curve) - 1, curve_edges + 3 * len(base_cosine_curve) - 1))

visualization.plot_line_segments(total_curves, total_edges, color = 'black')
low = -2.5
high = 7.5
plt.plot([low, low], [low, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([low, high], [low, low], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([low, high], [high, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.plot([high, high], [low, high], linestyle='--', dashes=(5, 5), c = 'black')
plt.axis('off')
plt.savefig('cosine_curves.svg', dpi = 300, bbox_inches='tight')

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
name = 'cosine_curve'

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.recordStart("{}_inflation.mp4".format(name), outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
import experiment_helper

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
# ipu.setVars(np.load('parallel_tubes.npy'))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()
benchmark.report()

In [ ]:
viewer.recordStop()

### Pattern 4: Elliptic holes

In [ ]:
time = '2024_01_17_11_33'
name = 'square_with_ellipse_hole_angle_width_height'
experiment_file = '../../experiments/parallelized_experiments/output/{}/{}/experiment_result.json'.format(name, time)
stiffness_path = '../../experiments/parallelized_experiments/output/{}/{}'.format(name, time)

In [ ]:
label = '42.00_0.20_1.30'

m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/{}/{}/{}/mesh_{}_{}.obj'.format(name, time, label, name, label))
fusing_vtx = np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/fusedVtx_{}_{}.npy'.format(name, time, label, name, label))

visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)
vertices = m.vertices()
vertices = np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1)
m = MeshFEM.mesh.Mesh(vertices, m.elements())
# fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusing_vtx, epsilon = 1e-9)
# ipu.setVars(np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/{}_dofs_before_stiffness_{}.npy'.format(name, time, label, name, label)))
viewer = TriMeshViewer(ipu, width=500, height=500)
viewer.showWireframe(False)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
name = 'elliptic_holes'

In [ ]:
import tri_mesh_viewer

In [ ]:
# viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
viewer.showWireframe(True)


In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.recordStart("{}_inflation.mp4".format(name), outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
import experiment_helper

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
# ipu.setVars(np.load('parallel_tubes.npy'))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()
benchmark.report()

In [ ]:
ipu.energy()

In [ ]:
curr_vars = ipu.getVars()

In [ ]:
curr_vars[-2] = 1e-1
curr_vars[-1] = np.pi / 2

In [ ]:
ipu.setVars(curr_vars)

In [ ]:
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6


In [ ]:
cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()

### Pattern 5: isotrpic three star

In [ ]:
a = 2
avg_len = 0.08
radius = 0.8
ipu, m, marker = pattern_generator_using_gmsh.get_three_star_hex(a, radius, num_spikes=3, avg_len_boundary = avg_len, avg_len_embeddings = avg_len)        
finalMarkers = np.where(np.array(marker) == 1)[0]

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
base_curve, curve_edges, boundary_vxs, boundary_lines = pattern_generator_using_gmsh.get_three_star_hex(a, radius, num_spikes=3, avg_len_boundary = avg_len, avg_len_embeddings = avg_len, return_line_segments = True)      

total_curves = base_curve
total_edges = curve_edges - 1

visualization.plot_line_segments(total_curves, total_edges, color = 'black')
low = 0
high = 8
plt.plot(list(boundary_vxs[:, 0]) + [boundary_vxs[0][0]], list(boundary_vxs[:, 1]) + [boundary_vxs[0][1]], linestyle='--', dashes=(5, 5), c = 'black')
plt.axis('off')
plt.savefig('three_star.svg', dpi = 300, bbox_inches='tight')

In [ ]:
boundary_vxs

In [ ]:
boundary_vxs

In [ ]:
visualization.plot_2d_mesh(m, pointList=fusedVtx)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
name = 'three_stars'

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
viewer.recordStart("{}_inflation.mp4".format(name), outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
import experiment_helper

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
# ipu.setVars(np.load('parallel_tubes.npy'))

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()
benchmark.report()

In [ ]:
import periodic_simulation_setup

In [ ]:
ipu.setVars(ipu.getVars() + 1e-3 * np.random.random(ipu.numVars()))

In [ ]:
az_ipu = periodic_simulation_setup.get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT)

In [ ]:
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(False)
az_viewer.show()

In [ ]:
def az_cb(i):
    az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])
    # az_viewer.update()

In [ ]:
az_cb(0)

In [ ]:

opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, az_ipu.getBendingStiffnessFixedVars(), opts, callback=az_cb, hessianShift = 1e-7)
cr = az_optimizer.optimize()

In [ ]:
az_ipu.energy()

In [ ]:
az_ipu.energy()

In [ ]:
curr_vars = az_ipu.getVars()

In [ ]:
curr_vars[-2] = 1e-1
curr_vars[-1] = np.pi / 2

In [ ]:
az_ipu.setVars(curr_vars)

In [ ]:
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6


In [ ]:
cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()

In [ ]:
ipu.energy()

In [ ]:
result_folder = '.'
variable = 0
render_images = True

In [ ]:
az_ipu.getBendingStiffnessFixedVars()

In [ ]:
stiffness_values, sampled_alphas, stiffness_coefficient = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = 0, fixedVars = az_ipu.getBendingStiffnessFixedVars(), filename = None if (result_folder is None) else ("{}/stiffness_{}_{}.png".format(result_folder, name, variable)), generate_images = render_images)

In [ ]:
from IPython.display import Image
Image(filename='stiffness_three_stars_0.png') 

In [ ]:
ipu.energy()

In [ ]:
curr_vars = ipu.getVars()

In [ ]:
curr_vars[-2] = 1e-1
curr_vars[-1] = np.pi / 2

In [ ]:
ipu.setVars(curr_vars)

In [ ]:
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6


In [ ]:
cr = inflation.inflation_newton(ipu, fixedVars, options = opts, callback=cb, hessianShift = hessianShift)
# viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.update()

In [ ]:
ipu.energy()

In [ ]:
viewer.recordStop()

### Jus a plane

In [ ]:
import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

import sys; sys.path.append("../../"); sys.path.append("../../../.."); sys.path.append("../../../gmsh/"); sys.path.append("../../experiments/"); sys.path.append("../../../")
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:
stiffness_data_collection = []

In [ ]:
hessianShiftForStiffness = 1e-7


In [ ]:
angle = 90
r = 2.5
avg_len = 0.04


h = 5
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])
m, marker = pattern_generator_using_gmsh.get_single_dash(h, avg_len, avg_len, dash_point = dash_point)

finalMarkers = np.where(np.array(marker) == 1)[0]
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

# vertices = m.vertices()
# vertices *= 4
# m = MeshFEM.mesh.Mesh(vertices, m.elements())
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(False)
viewer.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Change this value to adjust the gravitational acceleration
g = -9.81 / 100
reduce_factor = 0.6
time_step = 0.5
steps = int(300 * 0.05 / time_step)
print(steps)
time_duration = 400
t = np.linspace(0, time_duration, steps, dtype=np.float64)
v = np.zeros(steps)
y = np.zeros(steps)
bounces = 0
y[0] = 1
final_i = 0

for i in range(1, steps):
    if y[i-1] + time_step * v[i-1] > 0:
        v[i] = v[i-1] + g * time_step
        y[i] = y[i-1] + time_step* v[i-1]
    if y[i-1] + time_step * v[i-1] <= 0:
        v[i] = -reduce_factor * v[i-1]
        y[i] = 0
        bounces += 1
    if bounces == 2:
        final_i = i
        break
print(final_i)
t = t[:final_i]
y = y[:final_i]

In [ ]:
plt.plot(t, y)

In [ ]:
y

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
viewer.setCameraParams(((0, -6, 4),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
steps = int(len(y) / 2)

In [ ]:
normal_time = np.linspace(0, -1, steps)

In [ ]:
cubic_time = np.linspace(0, -1, steps) ** 3

In [ ]:
pause_frame = int(steps / 5)
bounce_steps = int(steps / 2)

In [ ]:
# cubic_time, np.linspace(normal_time[-1], normal_time[-bounce_steps], int(bounce_steps / 1.5))**3, np.linspace(normal_time[-bounce_steps], normal_time[-1], int(bounce_steps / 1.5)) ** 3

In [ ]:
kappas = np.concatenate((y - 1, np.ones(int(len(y) /1.5)) * cubic_time[-1], cubic_time[::-1]))

In [ ]:
import time

In [ ]:
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 + np.pi / 8
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 - np.pi / 3
    ipu.setVars(curr_vars)
    viewer.update()

In [ ]:
viewer.recordStart("bending_deformation.mp4", outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 + np.pi / 8
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 - np.pi / 3
    ipu.setVars(curr_vars)
    viewer.update()

In [ ]:
viewer.recordStop()

### Perturbing the elliptic holes

In [ ]:
time = '2024_01_17_11_33'
name = 'square_with_ellipse_hole_angle_width_height'
experiment_file = '../../experiments/parallelized_experiments/output/{}/{}/experiment_result.json'.format(name, time)
stiffness_path = '../../experiments/parallelized_experiments/output/{}/{}'.format(name, time)

In [ ]:
label = '42.00_0.20_1.30'

m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/{}/{}/{}/mesh_{}_{}.obj'.format(name, time, label, name, label))
fusing_vtx = np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/fusedVtx_{}_{}.npy'.format(name, time, label, name, label))

visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)
vertices = m.vertices()
vertices = np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1)
m = MeshFEM.mesh.Mesh(vertices, m.elements())
# fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusing_vtx, epsilon = 1e-9)
ipu.setVars(np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/{}_dofs_before_stiffness_{}.npy'.format(name, time, label, name, label)))

In [ ]:
curr_vars = ipu.getVars()
curr_vars[3:-2] = (ipu.getVars()[3:-2].reshape((-1, 3)) + np.array([-5, -5, 0])).flatten()
ipu.setVars(curr_vars)

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
name = 'elliptic_holes'

In [ ]:
import tri_mesh_viewer

In [ ]:
viewer = tri_mesh_viewer.OffscreenTriMeshViewer(ipu, width = 1000, height = 1000)

In [ ]:
viewer = TriMeshViewer(ipu, width=500, height=500)
viewer.showWireframe(False)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.show()

In [ ]:
def cb(i):
    # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    viewer.update()

In [ ]:
viewer.setCameraParams(((0, -4, 5),
 (0.0, 1, 0),
 (0.0, 0.0, 0.0)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Change this value to adjust the gravitational acceleration
g = -9.81 / 100
reduce_factor = 0.6
time_step = 0.5
steps = int(300 * 0.05 / time_step)
print(steps)
time_duration = 400
t = np.linspace(0, time_duration, steps, dtype=np.float64)
v = np.zeros(steps)
y = np.zeros(steps)
bounces = 0
y[0] = 1
final_i = 0

for i in range(1, steps):
    if y[i-1] + time_step * v[i-1] > 0:
        v[i] = v[i-1] + g * time_step
        y[i] = y[i-1] + time_step* v[i-1]
    if y[i-1] + time_step * v[i-1] <= 0:
        v[i] = -reduce_factor * v[i-1]
        y[i] = 0
        bounces += 1
    if bounces == 2:
        final_i = i
        break
print(final_i)
t = t[:final_i]
y = y[:final_i]

In [ ]:
plt.plot(t, y)

In [ ]:
steps = int(len(y) / 2)

In [ ]:
normal_time = np.linspace(0, -1, steps)

In [ ]:
cubic_time = np.linspace(0, -1, steps) ** 3

In [ ]:
pause_frame = int(steps / 5)
bounce_steps = int(steps / 2)

In [ ]:
# cubic_time, np.linspace(normal_time[-1], normal_time[-bounce_steps], int(bounce_steps / 1.5))**3, np.linspace(normal_time[-bounce_steps], normal_time[-1], int(bounce_steps / 1.5)) ** 3

In [ ]:
kappas = np.concatenate((y - 1, np.ones(int(len(y) /1.5)) * cubic_time[-1], cubic_time[::-1]))
kappas *= 0.2

In [ ]:
import time

In [ ]:
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 + np.pi / 8
    ipu.setVars(curr_vars)
    viewer.update()
    
for kappa in kappas:
    curr_vars  = ipu.getVars()
    curr_vars[-2] = kappa
    curr_vars[-1] = np.pi / 2 - np.pi / 3
    ipu.setVars(curr_vars)
    viewer.update()

In [ ]:
viewer.recordStart("elliptic_holes_bending.mp4", outWidth = 1000, outHeight = 1000, streaming = True)

In [ ]:
viewer.recordStop()